In [ ]:
# In[1]:

import pandas as pd
import numpy as np

# load metric.csv and parse timestamps as UTC
df = pd.read_csv("metric.csv")
df['timestamp'] = pd.to_datetime(df['timestamp'], unit='s', utc=True)

# compute global P95 and total points in full series per (cmdb_id, kpi_name) BEFORE any window filtering
p95_df = df.groupby(['cmdb_id', 'kpi_name'])['value'].agg(
    total_points_in_full_series='count',
    global_p95=lambda x: float(np.nanpercentile(x.values, 95))
).reset_index()

# define the incident window (UTC)
window_start = pd.to_datetime("2024-01-18 20:47:34", utc=True)
window_end = pd.to_datetime("2024-01-18 21:17:34", utc=True)

# filter rows to the window
df_window = df[(df['timestamp'] >= window_start) & (df['timestamp'] <= window_end)].copy()

# total points in window per group
window_counts = df_window.groupby(['cmdb_id', 'kpi_name']).size().reset_index(name='total_points_in_window')

# join global p95 info to the window rows to evaluate above-P95
df_win_with_p95 = df_window.merge(p95_df, on=['cmdb_id', 'kpi_name'], how='left')

# rows in window with value > global_p95
df_above = df_win_with_p95[df_win_with_p95['value'] > df_win_with_p95['global_p95']].copy()

# aggregated above-P95 info per group: count and earliest timestamp
above_agg = df_above.groupby(['cmdb_id', 'kpi_name']).agg(
    count_points_in_window_above_p95=('value', 'count'),
    earliest_timestamp_in_window_above_p95=('timestamp', 'min')
).reset_index()

# assemble final summary by merging p95_df, window_counts, and above_agg
summary = p95_df.merge(window_counts, on=['cmdb_id', 'kpi_name'], how='left') \
                .merge(above_agg, on=['cmdb_id', 'kpi_name'], how='left')

# fill NaNs for groups with no points in window / no above-p95 hits
summary['total_points_in_window'] = summary['total_points_in_window'].fillna(0).astype(int)
summary['count_points_in_window_above_p95'] = summary['count_points_in_window_above_p95'].fillna(0).astype(int)
# format earliest timestamp as ISO string in UTC, keep empty if NaT
summary['earliest_timestamp_in_window_above_p95'] = summary['earliest_timestamp_in_window_above_p95'].dt.strftime('%Y-%m-%dT%H:%M:%SZ')
summary['earliest_timestamp_in_window_above_p95'] = summary['earliest_timestamp_in_window_above_p95'].fillna('')

# select and order columns as requested
summary = summary[
    ['cmdb_id', 'kpi_name',
     'total_points_in_full_series', 'total_points_in_window',
     'global_p95', 'count_points_in_window_above_p95',
     'earliest_timestamp_in_window_above_p95']
]

# sort by count_points_in_window_above_p95 desc and show up to top 20 rows
summary = summary.sort_values(by='count_points_in_window_above_p95', ascending=False).head(20)

# display the compact summary table
summary

```
Out[1]:
```
```python
# Summarize the previously computed `summary` DataFrame into a plain-English conclusion string.
# Reuse existing `summary` variable from the previous cell.

# compact stats
top_hits = summary[summary['count_points_in_window_above_p95'] > 0].copy()
total_groups_exceeding = top_hits.shape[0]

# component-level counts (how many distinct KPIs per component exceeded P95 in the window)
comp_counts = top_hits.groupby('cmdb_id').size().sort_values(ascending=False)

# top component details
top_component = comp_counts.index[0] if not comp_counts.empty else None
top_component_kpis = top_hits[top_hits['cmdb_id'] == top_component]['kpi_name'].tolist() if top_component else []
top_component_earliest = top_hits[top_hits['cmdb_id'] == top_component]['earliest_timestamp_in_window_above_p95'].min() if top_component else ''

# other impacted components (name:count)
other_comps = ", ".join(f"{c}({n})" for c, n in comp_counts.items() if c != top_component)

summary_str = (
    f"Summary of results for window 2024-01-18 20:47:34 to 2024-01-18 21:17:34 UTC:\n"
    f"- {total_groups_exceeding} metric series (cmdb_id, kpi_name) exceeded their global P95 during the window.\n"
    f"- The top affected component is '{top_component}' with {comp_counts.iloc[0]} distinct KPIs exceeding P95: {', '.join(top_component_kpis)}.\n"
    f"  Earliest exceedance for '{top_component}' in the window: {top_component_earliest}.\n"
    f"- Other affected components and their counts: {other_comps if other_comps else 'none'}.\n"
    f"- Each listed group had 2 points above the global P95 in the window (as shown in the table).\n\n"
    f"Interpretation: 'carts' is the most likely impacted service based on multiple distinct KPI series exceeding their global P95. "
    f"Investigate carts' CPU, memory, latency and socket-related metrics around the listed earliest timestamps."
)

summary_str
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id    kpi_name  total_points_in_full_series  total_points_in_window    global_p95  count_points_in_window_above_p95 earliest_timestamp_in_window_above_p95
0          carts         cpu                           25                      25  1.991059e+00                                 2                   2024-01-18T21:11:00Z
2          carts  latency-50                           25                      25  4.071341e-02                                 2                   2024-01-18T21:11:00Z
4          carts         mem                           25                      25  2.168275e+08                                 2                   2024-01-18T21:13:00Z
3          carts  latency-90                           25                      25  1.676573e-01                                 2                   2024-01-18T21:11:00Z
5          carts      socket                           25                      25  1.627857e+01                                 2                   2024-01-18T21:11:00Z
6          carts    workload                           25                      25  8.447293e+00                                 2                   2024-01-18T20:53:00Z
11     catalogue         cpu                           25                      25  1.849698e-01                                 2                   2024-01-18T20:54:00Z
7       carts-db         cpu                           25                      25  1.278996e+01                                 2                   2024-01-18T21:13:00Z
8       carts-db      diskio                           25                      25  2.636501e+06                                 2                   2024-01-18T20:55:00Z
9       carts-db         mem                           25                      25  1.323617e+08                                 2                   2024-01-18T21:13:00Z
13     catalogue  latency-50                           25                      25  3.055612e-03                                 2                   2024-01-18T20:50:00Z
17     catalogue    workload                           25                      25  4.251860e+00                                 2                   2024-01-18T20:54:00Z
15     catalogue         mem                           25                      25  6.327165e+06                                 2                   2024-01-18T21:12:00Z
14     catalogue  latency-90                           25                      25  4.700102e-03                                 2                   2024-01-18T20:50:00Z
76       user-db         mem                           25                      25  1.063630e+08                                 2                   2024-01-18T21:13:00Z
71          user         mem                           25                      25  8.778001e+06                                 2                   2024-01-18T21:12:00Z
73          user    workload                           25                      25  1.913574e+01                                 2                   2024-01-18T20:53:00Z
18  catalogue-db         cpu                           25                      25  2.165697e-01                                 2                   2024-01-18T20:54:00Z
20  catalogue-db         mem                           25                      25  4.324558e+08                                 2                   2024-01-18T21:00:00Z
24     front-end  latency-50                           25                      25  3.783974e-02                                 2                   2024-01-18T20:54:00Z```
```

In [ ]:
# In[2]:

import pandas as pd
import numpy as np

# load trace.csv and parse timestamps as UTC (reuse pd, np)
trace_df = pd.read_csv("trace.csv")
trace_df['timestamp'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True)

# compute global P95 and total points in full series per (cmdb_id, trace_name) BEFORE window filtering
p95_trace_df = trace_df.groupby(['cmdb_id', 'trace_name'])['value'].agg(
    total_points_in_full_series='count',
    global_p95=lambda x: float(np.nanpercentile(x.values, 95))
).reset_index()

# define the incident window (UTC)
window_start = pd.to_datetime("2024-01-18 20:47:34", utc=True)
window_end = pd.to_datetime("2024-01-18 21:17:34", utc=True)

# filter rows to the window
trace_window = trace_df[(trace_df['timestamp'] >= window_start) & (trace_df['timestamp'] <= window_end)].copy()

# total points in window per group
window_counts = trace_window.groupby(['cmdb_id', 'trace_name']).size().reset_index(name='total_points_in_window')

# join global p95 info to the window rows to evaluate above-P95
trace_win_with_p95 = trace_window.merge(p95_trace_df, on=['cmdb_id', 'trace_name'], how='left')

# rows in window with value > global_p95
trace_above = trace_win_with_p95[trace_win_with_p95['value'] > trace_win_with_p95['global_p95']].copy()

# aggregated above-P95 info per group: count and earliest timestamp
above_agg = trace_above.groupby(['cmdb_id', 'trace_name']).agg(
    count_points_in_window_above_p95=('value', 'count'),
    earliest_timestamp_in_window_above_p95=('timestamp', 'min')
).reset_index()

# assemble final summary by merging p95_trace_df, window_counts, and above_agg
trace_summary = p95_trace_df.merge(window_counts, on=['cmdb_id', 'trace_name'], how='left') \
                .merge(above_agg, on=['cmdb_id', 'trace_name'], how='left')

# fill NaNs for groups with no points in window / no above-p95 hits
trace_summary['total_points_in_window'] = trace_summary['total_points_in_window'].fillna(0).astype(int)
trace_summary['count_points_in_window_above_p95'] = trace_summary['count_points_in_window_above_p95'].fillna(0).astype(int)
# format earliest timestamp as ISO string in UTC, keep empty if NaT
trace_summary['earliest_timestamp_in_window_above_p95'] = trace_summary['earliest_timestamp_in_window_above_p95'].dt.strftime('%Y-%m-%dT%H:%M:%SZ')
trace_summary['earliest_timestamp_in_window_above_p95'] = trace_summary['earliest_timestamp_in_window_above_p95'].fillna('')

# select and order requested columns
trace_summary = trace_summary[
    ['cmdb_id', 'trace_name',
     'total_points_in_full_series', 'total_points_in_window',
     'global_p95', 'count_points_in_window_above_p95',
     'earliest_timestamp_in_window_above_p95']
]

# sort by count_points_in_window_above_p95 desc and show up to top 20 rows
trace_summary = trace_summary.sort_values(by='count_points_in_window_above_p95', ascending=False).head(20)

# display the compact summary table
trace_summary

```
Out[2]:
```
# Summarize the trace analysis result as a plain-English string.
summary = (
    "Summary of trace analysis for window 2024-01-18 20:47:34 to 2024-01-18 21:17:34 UTC: "
    "No trace metric series (cmdb_id, trace_name) had any data points in the incident window, "
    "so none exceeded their global P95. "
    "Recommendation: verify trace ingestion for that interval or rely on metric/log telemetry for root-cause investigation."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

Empty DataFrame
Columns: [cmdb_id, trace_name, total_points_in_full_series, total_points_in_window, global_p95, count_points_in_window_above_p95, earliest_timestamp_in_window_above_p95]
Index: []```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np

# load log.csv and parse timestamps as UTC (reuse pd, np)
log_df = pd.read_csv("log.csv")
log_df['timestamp'] = pd.to_datetime(log_df['timestamp'], unit='s', utc=True)

# compute global P95 and total points in full series per (cmdb_id, log_name) BEFORE window filtering
p95_logs = log_df.groupby(['cmdb_id', 'log_name'])['value'].agg(
    total_points_in_full_series='count',
    global_p95=lambda x: float(np.nanpercentile(x.values, 95))
).reset_index()

# define the incident window (UTC)
window_start = pd.to_datetime("2024-01-18 20:47:34", utc=True)
window_end = pd.to_datetime("2024-01-18 21:17:34", utc=True)

# filter log rows to the window
log_window = log_df[(log_df['timestamp'] >= window_start) & (log_df['timestamp'] <= window_end)].copy()

# total points in window per group
window_counts = log_window.groupby(['cmdb_id', 'log_name']).size().reset_index(name='total_points_in_window')

# join global p95 info to the window rows to evaluate above-P95
log_win_with_p95 = log_window.merge(p95_logs, on=['cmdb_id', 'log_name'], how='left')

# rows in window with value > global_p95
log_above = log_win_with_p95[log_win_with_p95['value'] > log_win_with_p95['global_p95']].copy()

# aggregated above-P95 info per group: count and earliest timestamp
above_agg = log_above.groupby(['cmdb_id', 'log_name']).agg(
    count_points_in_window_above_p95=('value', 'count'),
    earliest_timestamp_in_window_above_p95=('timestamp', 'min')
).reset_index()

# assemble final summary by merging p95_logs, window_counts, and above_agg
log_summary = p95_logs.merge(window_counts, on=['cmdb_id', 'log_name'], how='left') \
                .merge(above_agg, on=['cmdb_id', 'log_name'], how='left')

# fill NaNs for groups with no points in window / no above-p95 hits
log_summary['total_points_in_window'] = log_summary['total_points_in_window'].fillna(0).astype(int)
log_summary['count_points_in_window_above_p95'] = log_summary['count_points_in_window_above_p95'].fillna(0).astype(int)
# format earliest timestamp as ISO string in UTC, keep empty if NaT
log_summary['earliest_timestamp_in_window_above_p95'] = pd.to_datetime(log_summary['earliest_timestamp_in_window_above_p95']).dt.strftime('%Y-%m-%dT%H:%M:%SZ')
log_summary['earliest_timestamp_in_window_above_p95'] = log_summary['earliest_timestamp_in_window_above_p95'].fillna('')

# select and order requested columns
log_summary = log_summary[
    ['cmdb_id', 'log_name',
     'total_points_in_full_series', 'total_points_in_window',
     'global_p95', 'count_points_in_window_above_p95',
     'earliest_timestamp_in_window_above_p95']
]

# sort by count_points_in_window_above_p95 desc and show up to top 20 rows
log_summary = log_summary.sort_values(by='count_points_in_window_above_p95', ascending=False).head(20)


# ---------------------------
# Process error_logs.csv
# ---------------------------
err_df = pd.read_csv("error_logs.csv")
err_df['timestamp'] = pd.to_datetime(err_df['timestamp'], unit='s', utc=True)

# filter to the incident window
err_window = err_df[(err_df['timestamp'] >= window_start) & (err_df['timestamp'] <= window_end)].copy()

if err_window.empty:
    # create empty summary with the required columns
    error_summary = pd.DataFrame(columns=[
        'cmdb_id', 'total_error_messages_in_window', 'earliest_error_timestamp_in_window', 'sample_messages'
    ])
else:
    # aggregated counts and earliest timestamp per cmdb_id
    err_agg = err_window.groupby('cmdb_id').agg(
        total_error_messages_in_window=('message', 'count'),
        earliest_error_timestamp_in_window=('timestamp', 'min')
    ).reset_index()

    # sample up to 5 distinct messages per cmdb_id (deduplicated), joined by ' || '
    sample_msgs = err_window.groupby('cmdb_id')['message'].apply(
        lambda s: ' || '.join(pd.Series(s).dropna().astype(str).unique()[:5])
    ).reset_index(name='sample_messages')

    error_summary = err_agg.merge(sample_msgs, on='cmdb_id', how='left')

    # format earliest timestamp as ISO string in UTC
    error_summary['earliest_error_timestamp_in_window'] = pd.to_datetime(error_summary['earliest_error_timestamp_in_window']).dt.strftime('%Y-%m-%dT%H:%M:%SZ')
    error_summary = error_summary[['cmdb_id', 'total_error_messages_in_window', 'earliest_error_timestamp_in_window', 'sample_messages']]

# display the two compact summary tables
log_summary, error_summary

```
Out[3]:
```
Summary of log and error-log analysis for 2024-01-18 20:47:34 — 2024-01-18 21:17:34 UTC:

- Top anomalies (log.row_count above their global P95):
  - front-end: 2 points above P95, earliest exceedance at 2024-01-18T20:53:00Z.
  - orders: 2 points above P95, earliest exceedance at 2024-01-18T21:04:00Z.
  - user: 2 points above P95, earliest exceedance at 2024-01-18T20:53:00Z.
- Additional single exceedances (1 point above P95) observed for:
  - carts (earliest 2024-01-18T21:10:00Z), queue-master (20:53:00Z), shipping (20:53:00Z), catalogue (20:53:00Z), carts-db (21:10:00Z).
- log.error_count series did not show any counts above their P95 (no spike in error counts).
- error_logs.csv contained no error messages in the incident window.

Interpretation and recommended next steps:
- The logs point to a traffic/workload spike affecting front-end, orders, and user services (and one-off spikes in carts and several supporting services), rather than an increase in logged errors.
- Investigate request rates/throughput and latencies for front-end → orders → user → carts paths around the listed timestamps, and check related resource metrics (CPU, memory, disk I/O) for those services.

The original code execution output of IPython Kernel is also provided below for reference:

(         cmdb_id         log_name  total_points_in_full_series  total_points_in_window  global_p95  count_points_in_window_above_p95 earliest_timestamp_in_window_above_p95
7      front-end    log.row_count                           25                      25      2021.8                                 2                   2024-01-18T20:53:00Z
9         orders    log.row_count                           25                      25      1824.0                                 2                   2024-01-18T21:04:00Z
21          user    log.row_count                           25                      25       616.2                                 2                   2024-01-18T20:53:00Z
1          carts    log.row_count                           24                      24       125.0                                 1                   2024-01-18T21:10:00Z
15  queue-master    log.row_count                           21                      21       576.0                                 1                   2024-01-18T20:53:00Z
19      shipping    log.row_count                           21                      21        64.0                                 1                   2024-01-18T20:53:00Z
5      catalogue    log.row_count                           25                      25       169.0                                 1                   2024-01-18T20:53:00Z
3       carts-db    log.row_count                           24                      24        13.0                                 1                   2024-01-18T21:10:00Z
6      front-end  log.error_count                           25                      25         0.0                                 0                                       
4      catalogue  log.error_count                           25                      25         0.0                                 0                                       
0          carts  log.error_count                           24                      24         0.0                                 0                                       
2       carts-db  log.error_count                           24                      24         0.0                                 0                                       
11     orders-db    log.row_count                            1                       1         1.0                                 0                                       
12       payment  log.error_count                           21                      21         0.0                                 0                                       
10     orders-db  log.error_count                            1                       1         0.0                                 0                                       
8         orders  log.error_count                           25                      25         0.0                                 0                                       
14  queue-master  log.error_count                           21                      21         0.0                                 0                                       
13       payment    log.row_count                           21                      21       104.0                                 0                                       
17    session-db    log.row_count                            4                       4         5.0                                 0                                       
16    session-db  log.error_count                            4                       4         0.0                                 0                                       , Empty DataFrame
Columns: [cmdb_id, total_error_messages_in_window, earliest_error_timestamp_in_window, sample_messages]
Index: [])```
```